# pdbe_sifts — Quickstart

This notebook walks through the complete SIFTS pipeline:

| Step | What happens |
|------|--------------|
| 0 | Install dependencies |
| 1 | Download Swiss-Prot & build the taxonomy map |
| 2 | Initialise the configuration |
| 3 | Build the MMseqs2 reference database |
| 4 | Run global mappings on a PDB entry |
| 5 | Explore the results in DuckDB |
| 6 | Segment generation — DuckDB mode |
| 7 | Segment generation — FASTA mode (`-m`) |
| 8 | Load segments into DuckDB (`db_load`) |

Each step shows both the **Python API** and the equivalent **CLI command** as a comment.

---
## 0 · Installation

**Requirements**
- Python ≥ 3.10
- [MMseqs2](https://github.com/soedinglab/MMseqs2) — fast sequence search
- [FASTA36](https://fasta.bioch.virginia.edu/wrpearson/fasta/) (`lalign36`) — local pairwise alignment

Run the cells below only if you have not already set up the environment.

In [1]:
%%bash
# From the repository root
conda env create -f ../environment.yml
conda activate pdbe_sifts
pip install -e ..

bash: line 2: conda: command not found
bash: line 3: conda: command not found


Obtaining file:///Users/adamb/Documents/pdbe_sifts
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build editable did not run successfully.
  │ exit code: 1
  ╰─> [43 lines of output]
      Traceback (most recent call last):
        File "/Users/adamb/micromamba/envs/pdbe_sifts/lib/python3.14/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "/Users/adamb/micromamba/envs/pdbe_sifts/lib/python3.14/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "/Users/adamb/micromamba/envs/pdbe_sifts/lib/python3.14/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 157, in get_requires_for_build_editable
          return hook(config_settings)
        File "/private/var/folders/0p/tq62kqzx2kg6lmgynvf6yw9w0000gp/T/pip-build-env-qa02b7e9/overl

CalledProcessError: Command 'b'# From the repository root\nconda env create -f ../environment.yml\nconda activate pdbe_sifts\npip install -e ..\n'' returned non-zero exit status 1.

In [ ]:
%%bash
conda install -c conda-forge mmseqs2
conda install -c bioconda fasta3    # provides lalign36

In [2]:
# Verify that all required tools are available
import shutil

for tool in ["pdbe_sifts", "mmseqs", "lalign36"]:
    found = shutil.which(tool)
    print(f"{tool:15s} {'✓  ' + found if found else '✗  NOT FOUND'}")

pdbe_sifts      ✓  /Users/adamb/micromamba/envs/pdbe_sifts/bin/pdbe_sifts
mmseqs          ✓  /Users/adamb/micromamba/envs/pdbe_sifts/bin/mmseqs
lalign36        ✓  /Users/adamb/micromamba/envs/pdbe_sifts/bin/lalign36


---
## 1 · Swiss-Prot reference database

**Swiss-Prot** is the manually reviewed section of UniProt (~571 000 entries).
It is the recommended reference for the SIFTS pipeline because every entry has been
curated by expert biocurators.

Each FASTA header encodes the accession and NCBI taxonomy ID:

```
>sp|Q6GZX4|001R_FRG3G Putative transcription factor 001R OS=Frog virus 3 OX=654924 GN=FV3-001R PE=4 SV=1
MAFSAEDVLKEYDRRRRMEALLLSLYYPNDRKLLDYKEWSPP…
```

We need two things:
1. The FASTA file itself — for the sequence search database.
2. A two-column TSV `{accession}\t{taxid}` — for taxonomy-aware scoring.

In [3]:
# CLI equivalent:
# wget https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz

import urllib.request, os

FASTA = "uniprot_sprot.fasta.gz"
URL   = (
    "https://ftp.uniprot.org/pub/databases/uniprot/current_release/"
    "knowledgebase/complete/uniprot_sprot.fasta.gz"
)

if not os.path.exists(FASTA):
    print("Downloading Swiss-Prot (~280 MB)…")
    urllib.request.urlretrieve(URL, FASTA)
    print("Done.")
else:
    print(f"Found {FASTA}, skipping download.")

Found uniprot_sprot.fasta.gz, skipping download.


In [4]:
# Build sp2taxid.tsv — two columns: accession <tab> NCBI taxon ID
#
# CLI equivalent (bash):
# zcat uniprot_sprot.fasta.gz | grep '^>' | \
#   perl -ne 'if (/^>sp\|([^|]+)\|.*OX=(\d+)/) { print "$1\t$2\n" }' > sp2taxid.tsv

import gzip, re

with gzip.open(FASTA, "rt") as fasta, open("sp2taxid.tsv", "w") as out:
    for line in fasta:
        if not line.startswith(">sp|"):
            continue
        acc = line.split("|")[1]              # >sp|{acc}|...
        m   = re.search(r"OX=(\d+)", line)   # ... OX={taxid} ...
        if m:
            out.write(f"{acc}\t{m.group(1)}\n")

print("sp2taxid.tsv written.")

sp2taxid.tsv written.


---
## 2 · Configuration

`pdbe_sifts init` copies a default configuration template to
`~/.config/pdbe_sifts/config.yaml`.
You then need to set at minimum:

| Key | Description |
|-----|-------------|
| `user.base_dir` | Root directory for processed data |
| `user.nobackup_dir` | Fast scratch directory (caches, indices) |
| `user.target_db` | Path to the preformatted sequence database (set in step 3) |

In [9]:
# CLI equivalent:
# pdbe_sifts init
!pdbe_sifts init

Config template copied to: /Users/adamb/Library/Application Support/pdbe_sifts/config.yaml
Please edit it before running the pipeline.
2026-03-16 19:21:17 JRHHG9XHV9 root[15653] INFO Initializing NCBI taxonomy database (first run may download ~70 MB)...
2026-03-16 19:21:17 JRHHG9XHV9 root[15653] INFO NCBI taxonomy database ready.


In [10]:
# CLI equivalent:
# pdbe_sifts show
!pdbe_sifts show

{'user': {'base_dir': '/Users/adamb/Desktop/sifts/', 'nobackup_dir': '/Users/adamb/Desktop/sifts/tmp/', 'target_db': '/Users/adamb/Desktop/sifts/db/mmseqs_sp/sp.db'}, 'location': {'base': '${user.base_dir}', 'nobackup_dir': '${user.nobackup_dir}', 'work': {'base_dir': '${location.base}/release', 'data_dir': '${location.work.base_dir}/data', 'data_entry_dir': '${location.work.data_dir}/entry'}}, 'cache': {'base': '${location.nobackup_dir}/sifts_data_cache/', 'uniprot': '${cache.base}/uniprot/', 'ccd': '${cache.base}/ccd/'}, 'alignment': {'mmseqs': {'sensitivity': 5.7, 'min_seq_id': 0.9, 'alignment_mode': 3, 'db_load_mode': 2}, 'blastp': {'evalue': 10.0}}}


# Edit the config, for example:
```
user:
  base_dir: /Users/adamb/Desktop/sifts/
  nobackup_dir: /Users/adamb/Desktop/sifts/tmp/
  target_db: /Users/adamb/Desktop/sifts/db/mmseqs_sp/sp.db


location:
  base: ${user.base_dir}
  nobackup_dir: ${user.nobackup_dir}

  work:
    base_dir: ${location.base}/release
    data_dir: ${location.work.base_dir}/data
    data_entry_dir: ${location.work.data_dir}/entry


cache:
  base: ${location.nobackup_dir}/sifts_data_cache/
  uniprot: ${cache.base}/uniprot/
  ccd: ${cache.base}/ccd/


# ---------------------------------------------------------------------------
# Alignment tool parameters
# Modify here to tune sensitivity/speed trade-offs without touching the code.
# ---------------------------------------------------------------------------

alignment:
  mmseqs:
    sensitivity: 5.7        # float, 1–7.5  (higher = more sensitive, slower)
    min_seq_id: 0.9         # float, 0–1    (minimum sequence identity threshold)
    alignment_mode: 3       # int, 0–4      (3 = backtrack; required for qaln/taln)
    db_load_mode: 2         # int, 0–3      (2 = memory-mapped, fastest for repeated searches)

  blastp:
    evalue: 10.0            # float         (e-value cut-off for saving hits)
```

---
## 3 · Build the MMseqs2 sequence database

`TargetDb` wraps `mmseqs createdb` + `createtaxdb` + `createindex` into a single call.
This step is slow (minutes to hours depending on database size) but only needs to be done once.

The resulting database at `DB_PATH` contains:
- the compressed sequence index,
- the taxonomy mapping,
- the lookup index for fast search.

In [5]:
# CLI equivalent:
# pdbe_sifts build_db -i uniprot_sprot.fasta.gz -o sp.db -t sp2taxid.tsv --tool mmseqs --threads 8

from pdbe_sifts.global_mappings.target_database import TargetDb

DB_PATH = "./sp.db"

TargetDb(
    input_path       = FASTA,
    output_path      = DB_PATH,
    tax_mapping_file = "sp2taxid.tsv",
    tool             = "mmseqs",
    threads          = 8,
).run()

2026-03-17 22:36:06 JRHHG9XHV9 root[72574] INFO Processing the creation of the database from uniprot_sprot.fasta.gz. This could take many hours/days.



 Running pymmseqs command... 
✓ Database creation completed successfully
  Results saved to: /Users/adamb/Documents/pdbe_sifts/notebooks/sp.db

 Running pymmseqs command... 
✓ Create Taxonomy Database completed successfully
  Results saved to: /Users/adamb/Documents/pdbe_sifts/notebooks/sp.db

 Running pymmseqs command... 


2026-03-17 22:36:22 JRHHG9XHV9 root[72574] INFO Creation finished.
2026-03-17 22:36:22 JRHHG9XHV9 root[72574] INFO Creation process took: 16.67232799998601 seconds.
2026-03-17 22:36:22 JRHHG9XHV9 root[72574] INFO Database saved: sp.db


✓ Create Index completed successfully
  Results saved to: /Users/adamb/Documents/pdbe_sifts/notebooks/sp.db


# Edit the config if you didn't and add the path to sp.db

---
## 4 · Global mappings — 1CBS

`SiftsGlobalMappings` runs the following for each polypeptide chain in the mmCIF file:
1. Extracts the entity sequence → FASTA
2. Searches the MMseqs2 database (≥ 90% identity by default)
3. Scores hits with the SIFTS scoring function:
   `sifts_score = adjusted_score + tax_score + dataset_score`
4. Stores the ranked table in **hits.duckdb**

We use **1CBS** (*Cellular retinoic acid-binding protein II*, human) as a simple mono-chain example.

In [5]:
# CLI equivalent:
# pdbe_sifts global_mappings -i ../tests/data/cif/1cbs.cif -o ./output --tool mmseqs

from pathlib import Path
from pdbe_sifts.sifts_global_mappings import SiftsGlobalMappings


CIF_FILE   = "../tests/data/cif/1cbs.cif"
OUTPUT_DIR = "./output"

pipeline = SiftsGlobalMappings(
    input_file = CIF_FILE,
    out_dir    = OUTPUT_DIR,
    db_file    = DB_PATH,
    tool       = "mmseqs",
)

# Build the result path deterministically from known attributes
# (entry_name is derived from the CIF filename; date is set at object creation)
result_dir = Path(OUTPUT_DIR) / f"{pipeline.tool}_{pipeline.entry_name}_{pipeline.date}"
db_path    = result_dir / "hits.duckdb"

pipeline.process()

print(f"Results : {result_dir}")
print(f"DuckDB  : {db_path}")

2026-03-17 17:15:27 JRHHG9XHV9 root[64550] INFO Processing [../tests/data/cif/1cbs.cif]
2026-03-17 17:15:27 JRHHG9XHV9 root[64550] INFO Results will be written to output/mmseqs_1cbs_17_17_03_2026
2026-03-17 17:15:27 JRHHG9XHV9 root[64550] INFO Extracting sequences from ../tests/data/cif/1cbs.cif
2026-03-17 17:15:27 JRHHG9XHV9 root[64550] INFO Processing the search of the query output/mmseqs_1cbs_17_17_03_2026/1cbs.fasta against sp.db.



 Running pymmseqs command... 


2026-03-17 17:15:29 JRHHG9XHV9 root[64550] INFO Search finished.
2026-03-17 17:15:29 JRHHG9XHV9 root[64550] INFO Search process took: 2.542533083993476 seconds.
2026-03-17 17:15:29 JRHHG9XHV9 root[64550] INFO Results saved: output/mmseqs_1cbs_17_17_03_2026/hits_1cbs.tsv
2026-03-17 17:15:29 JRHHG9XHV9 root[64550] INFO Parsing mmseqs hits.
2026-03-17 17:15:29 JRHHG9XHV9 root[64550] INFO Loading data with identity >= 0.9...
2026-03-17 17:15:29 JRHHG9XHV9 root[64550] INFO Loaded 4 hits passing identity filter


✓ Easy Search completed successfully
  Results saved to: /Users/adamb/Documents/pdbe_sifts/notebooks/output/mmseqs_1cbs_17_17_03_2026/hits_1cbs.tsv


2026-03-17 17:15:30 JRHHG9XHV9 root[64550] INFO Computing adjusted_scores...
2026-03-17 17:15:30 JRHHG9XHV9 root[64550] INFO Adjusted score computed.
2026-03-17 17:15:30 JRHHG9XHV9 root[64550] INFO Scoring 4 taxonomy id pairs using 10 workers
2026-03-17 17:15:32 JRHHG9XHV9 root[64550] INFO Updated 4 taxonomy score mappings in database
2026-03-17 17:15:32 JRHHG9XHV9 root[64550] INFO Taxonomy scoring complete!
2026-03-17 17:15:32 JRHHG9XHV9 root[64550] INFO Computing sifts_scores...
2026-03-17 17:15:32 JRHHG9XHV9 root[64550] INFO SIFTS score computed.
2026-03-17 17:15:32 JRHHG9XHV9 root[64550] INFO Parsed results ready at: output/mmseqs_1cbs_17_17_03_2026/hits.duckdb
2026-03-17 17:15:32 JRHHG9XHV9 root[64550] INFO Total time (extraction → ranked mappings): 5.49 s.


Results : output/mmseqs_1cbs_17_17_03_2026
DuckDB  : output/mmseqs_1cbs_17_17_03_2026/hits.duckdb


---
## 5 · Explore the hits

The `hits` table in `hits.duckdb` contains one row per (entry, entity, UniProt accession) candidate.
Key columns:

| Column | Description |
|--------|-------------|
| `accession` | UniProt accession |
| `identity` | Sequence identity (0–1) |
| `coverage` | Query coverage (0–1) |
| `adjusted_score` | `identity × coverage × (1 − mismatch/query_len) × 1000` |
| `tax_score` | Taxonomy concordance (NCBI hierarchy) |
| `dataset_score` | Provenance / annotation quality |
| `sifts_score` | Final SIFTS score (sum of the three above) |
| `hit_rank` | Dense rank per (entry, entity), 1 = best |

In [6]:
import duckdb

db_path = 'output/mmseqs_1cbs_17_17_03_2026/hits.duckdb'

conn = duckdb.connect(str(db_path), read_only=True)
df   = conn.execute("SELECT * FROM hits ORDER BY sifts_score DESC").df()
conn.close()

df[["entry", "entity", "accession", "query_tax_id", "target_tax_id", 
    "identity", "coverage", "sifts_score", "adjusted_score", 
    "tax_score", "dataset_score", "hit_rank"]]

,entry,entity,accession,query_tax_id,target_tax_id,identity,coverage,sifts_score,adjusted_score,tax_score,dataset_score,hit_rank
0,1cbs,1,P29373,9606,9606,1.000,1.0,1300.000000,1000.000000,200.0,100.0,1
1,1cbs,1,Q5PXY7,9606,9913,0.970,1.0,1001.678832,941.678832,0.0,60.0,2
2,1cbs,1,P51673,9606,10116,0.934,1.0,979.459854,879.459854,0.0,100.0,3
3,1cbs,1,P22935,9606,10090,0.934,1.0,972.642336,872.642336,0.0,100.0,4


In [7]:
# Best UniProt match per PDB entity
conn = duckdb.connect(str(db_path), read_only=True)
best = conn.execute("""
    SELECT   entry, entity, accession, identity, coverage,
             adjusted_score, tax_score, dataset_score, sifts_score
    FROM     hits
    WHERE    hit_rank = 1
    ORDER BY entity
""").df()
conn.close()

best

,entry,entity,accession,identity,coverage,adjusted_score,tax_score,dataset_score,sifts_score
0,1cbs,1,P29373,1.0,1.0,1000.0,200.0,100.0,1300.0


---
## 6 · Segment generation — DuckDB mode

`SiftsAlign` uses the best UniProt mapping from `hits.duckdb` to produce
residue- and segment-level CSV files for each polypeptide chain.

The pipeline per chain:
1. Aligns the PDB sequence against the UniProt sequence (`lalign36`)
2. Refines extended gaps using 3-D connectivity checks
3. Writes two gzipped CSVs: `{entry_id}_seg.csv.gz` and `{entry_id}_res.csv.gz`

```bash
# CLI equivalent:
# pdbe_sifts segments \
#   -i ../tests/data/cif/1cbs.cif \
#   -o ./output/segments_1cbs \
#   -d <hits.duckdb path>
```

In [8]:
from pathlib import Path
from pdbe_sifts.sifts_segments_generation import SiftsAlign

CIF_FILE      = "../tests/data/cif/1cbs.cif"
SEGMENTS_DIR  = "./output/segments_1cbs"

# db_path was set in Step 4 — reuse it here
sa = SiftsAlign(
    cif_file     = CIF_FILE,
    out_dir      = SEGMENTS_DIR,
    db_conn_str  = str(db_path),
)
sa.process_entry("1cbs")
if sa.conn:
    sa.conn.close()

print("Segment generation complete.")

2026-03-17 22:37:49 JRHHG9XHV9 root[72574] INFO Loading chem comp three-letter to one-letter mapping
2026-03-17 22:37:49 JRHHG9XHV9 root[72574] INFO No future revisions found
2026-03-17 22:37:49 JRHHG9XHV9 root[72574] INFO Processing [1cbs]
2026-03-17 22:37:53 JRHHG9XHV9 root[72574] INFO Loaded UNP pickle from /Users/adamb/Desktop/sifts/tmp/sifts_data_cache/uniprot/P/P29373/P29373.pkl
2026-03-17 22:37:53 JRHHG9XHV9 root[72574] INFO {'A': [SMapping(accession='P29373', range_start=2, range_stop=138)]}
2026-03-17 22:37:53 JRHHG9XHV9 root[72574] INFO Processing 1cbs chain A
2026-03-17 22:37:53 JRHHG9XHV9 root[72574] INFO Loaded UNP pickle from /Users/adamb/Desktop/sifts/tmp/sifts_data_cache/uniprot/P/P29373/P29373.pkl
2026-03-17 22:37:53 JRHHG9XHV9 root[72574] INFO Got UniProt for P29373
2026-03-17 22:37:53 JRHHG9XHV9 root[72574] INFO Valid mapping: P29373 2-138
2026-03-17 22:37:53 JRHHG9XHV9 root[72574] INFO Processing A: ['P29373']
1it [00:00,  3.93it/s]
1it [00:00, 1114.91it/s]
2026-03-

Segment generation complete.


In [ ]:
import gzip
import pandas as pd
from pdbe_sifts.segments_generation.generate_xref_csv import XRefSegment, XRefResidue

SEG_COLS = list(XRefSegment._fields)
RES_COLS = list(XRefResidue._fields)

seg_csv = next(Path(SEGMENTS_DIR).glob("*_seg.csv.gz"))
res_csv = next(Path(SEGMENTS_DIR).glob("*_res.csv.gz"))

with gzip.open(seg_csv, "rt") as f:
    df_seg = pd.read_csv(f, names=SEG_COLS)

with gzip.open(res_csv, "rt") as f:
    df_res = pd.read_csv(f, names=RES_COLS)

print(f"Segments : {len(df_seg)} rows")
print(f"Residues : {len(df_res)} rows\n")

display(df_seg)
display(df_res.head(10))

### Colonnes du CSV segment (`*_seg.csv.gz`)

| Colonne | Description |
|---------|-------------|
| `entry_id`, `entity_id`, `auth_asym_id`, `struct_asym_id` | Identifiants mmCIF |
| `accession`, `name`, `seq_version` | Accession UniProt, nom, version de séquence |
| `pdb_start` / `pdb_end` | Positions dans la séquence SEQRES PDB (1-based) |
| `unp_start` / `unp_end` | Positions correspondantes dans UniProt (1-based) |
| `auth_start` / `auth_end` + `_icode` | Numérotation auteur (avec insertion codes) |
| `unp_alignment`, `pdb_alignment` | Chaînes d'alignement brutes |
| `identity`, `score`, `conflicts`, `modifications` | Qualité de l'alignement |
| `best_mapping` | `True` = meilleur mapping UniProt pour cette chaîne |
| `canonical_acc` | `True` = accession canonique (pas d'isoforme) |
| `chimera` | `True` = la chaîne mappe vers plusieurs entrées UniProt |

### Colonnes du CSV résidu (`*_res.csv.gz`)

| Colonne | Description |
|---------|-------------|
| `entry_id`, `entity_id`, `auth_asym_id`, `struct_asym_id` | Identifiants mmCIF |
| `unp_segment_id` | FK vers le segment parent |
| `auth_seq_id`, `auth_seq_id_ins_code` | Numérotation auteur du résidu |
| `pdb_seq_id` | Position dans SEQRES (1-based) |
| `unp_seq_id` | Position dans UniProt (`None` si gap) |
| `observed` | `"Y"` si des coordonnées 3D existent |
| `pdb_one_letter_code`, `unp_one_letter_code` | Résidus PDB vs UniProt (1 lettre) |
| `chem_comp_id` | Code CCD à 3 lettres (`ALA`, `MSE`…) |
| `type` | Type de résidu (`Natural`, `Insertion`, `Modified`, `Chromophore`…) |
| `tax_id` | Taxon NCBI de l'organisme source |
| `best_mapping`, `canonical_acc` | Flags de sélection |
| `residue_id` | Identifiant unique : `{entry_id}_{struct_asym_id}_{pdb_seq_id}` |

---
## 7 · Segment generation — FASTA mode (`-m`)

When you already know the UniProt mapping (or want to use a custom sequence),
you can bypass the DuckDB lookup entirely by providing a FASTA file with the header:

```
>{entry_id}|{auth_asym_id}|{sequence_id}
```

- `entry_id` — PDB entry (e.g. `1cbs`)
- `auth_asym_id` — author chain ID (e.g. `A`)
- `sequence_id` — UniProt accession or any custom identifier (e.g. `P29373`)

The sequence in the FASTA is used directly for alignment, so it can be any protein
sequence — not only UniProt entries.

```bash
# CLI equivalent:
# pdbe_sifts segments \
#   -i ../tests/data/cif/1cbs.cif \
#   -o ./output/segments_1cbs_fasta \
#   -m ./output/mapping_1cbs.fasta
```

In [14]:
MAPPING_FASTA   = "./output/mapping_1cbs.fasta"
SEGMENTS_DIR_FM = "./output/segments_1cbs_fasta"

# Write the mapping FASTA — header format: >{entry_id}|{auth_asym_id}|{sequence_id}
Path(MAPPING_FASTA).parent.mkdir(parents=True, exist_ok=True)
Path(MAPPING_FASTA).write_text(
    ">1cbs|A|P29373\n"
    "MPNFSGNWKIIRSENFEELLKVLGVNVMLRKIAVAAASKPAVEIKQEGDTFYIKTSTTVRT"
    "TEINFKVGEEFEEQTVDGRPCKSLVKWESENKMVCEQKLLKGEGPKTSWTRELTNDGELIL"
    "TMTADDVVCTRVYVRE\n"
)

sa_fasta = SiftsAlign(
    cif_file = CIF_FILE,
    out_dir  = SEGMENTS_DIR_FM,
    unp_mode = MAPPING_FASTA,        # no DuckDB — FASTA mapping only
)
sa_fasta.process_entry("1cbs")

print("Segment generation (FASTA mode) complete.")

2026-03-17 22:50:29 JRHHG9XHV9 root[72574] INFO Loading chem comp three-letter to one-letter mapping
2026-03-17 22:50:29 JRHHG9XHV9 root[72574] INFO No future revisions found
2026-03-17 22:50:29 JRHHG9XHV9 root[72574] INFO Processing [1cbs]
2026-03-17 22:50:34 JRHHG9XHV9 root[72574] INFO Loaded 1 custom sequences from output/mapping_1cbs.fasta
2026-03-17 22:50:34 JRHHG9XHV9 root[72574] INFO {'A': [SMapping(accession='P29373', range_start=0, range_stop=0)]}
2026-03-17 22:50:34 JRHHG9XHV9 root[72574] INFO Processing 1cbs chain A
2026-03-17 22:50:34 JRHHG9XHV9 root[72574] INFO Valid mapping: P29373 0-0
2026-03-17 22:50:34 JRHHG9XHV9 root[72574] INFO Processing A: ['P29373']
1it [00:00, 12.74it/s]
1it [00:00, 19599.55it/s]
2026-03-17 22:50:34 JRHHG9XHV9 root[72574] INFO Segments generated
2026-03-17 22:50:34 JRHHG9XHV9 root[72574] INFO {'P29373': [([(1, 137)], [(2, 138)], <<class 'Bio.Align.MultipleSeqAlignment'> instance (2 records of length 137) at 13f41b110>)]}
2026-03-17 22:50:34 JRHHG

Segment generation (FASTA mode) complete.


In [15]:
!zless output/segments_1cbs_fasta/1cbs_seg.csv.gz

1cbs,1,1,A,A,P29373,P29373,0,2,1,138,137,1, ,137, ,0,,PNFSGNWKIIRSENFEELLKVLGVNVMLRKIAVAAASKPAVEIKQEGDTFYIKTSTTVRTTEINFKVGEEFEEQTVDGRPCKSLVKWESENKMVCEQKLLKGEGPKTSWTRELTNDGELILTMTADDVVCTRVYVRE,PNFSGNWKIIRSENFEELLKVLGVNVMLRKIAVAAASKPAVEIKQEGDTFYIKTSTTVRTTEINFKVGEEFEEQTVDGRPCKSLVKWESENKMVCEQKLLKGEGPKTSWTRELTNDGELILTMTADDVVCTRVYVRE,1.0,1.0,1,1,P29373,0


In [ ]:
import gzip
import pandas as pd
from pdbe_sifts.segments_generation.generate_xref_csv import XRefSegment, XRefResidue

SEG_COLS = list(XRefSegment._fields)
RES_COLS = list(XRefResidue._fields)

seg_csv_fm = next(Path(SEGMENTS_DIR_FM).glob("*_seg.csv.gz"))
res_csv_fm = next(Path(SEGMENTS_DIR_FM).glob("*_res.csv.gz"))

with gzip.open(seg_csv_fm, "rt") as f:
    df_seg_fm = pd.read_csv(f, names=SEG_COLS)

with gzip.open(res_csv_fm, "rt") as f:
    df_res_fm = pd.read_csv(f, names=RES_COLS)

print(f"Segments : {len(df_seg_fm)} rows")
print(f"Residues : {len(df_res_fm)} rows\n")

display(df_seg_fm)
display(df_res_fm.head(10))

---
## 8 · Charger les CSVs dans DuckDB (`db_load`)

`SiftsDB` crée deux tables et expose des méthodes pour charger les CSVs gzippés produits
aux étapes 6 et 7.

| Table | Fichier source |
|-------|---------------|
| `sifts_xref_segment` | `{entry_id}_seg.csv.gz` |
| `sifts_xref_residue` | `{entry_id}_res.csv.gz` |

> **Mode batch (production)** — `SiftsDB.bulk_load_from_entries(root_dir)` scanne
> l'arborescence `{root_dir}/*/*/sifts/` produite par `BatchSegmentGeneration`
> et charge toutes les entrées d'un coup via `read_csv` natif DuckDB.
>
> Ici on charge directement les fichiers produits à l'étape 6 via pandas.

In [ ]:
import gzip

import duckdb
import pandas as pd
from pdbe_sifts.database.sifts_db_wrapper import SiftsDB
from pdbe_sifts.segments_generation.generate_xref_csv import XRefSegment, XRefResidue

SEG_COLS = list(XRefSegment._fields)
RES_COLS = list(XRefResidue._fields)
SIFTS_DB  = "./output/sifts.duckdb"

# Read the CSVs produced at Step 6 (DuckDB mode)
with gzip.open(seg_csv, "rt") as f:
    df_seg_load = pd.read_csv(f, names=SEG_COLS)

with gzip.open(res_csv, "rt") as f:
    df_res_load = pd.read_csv(f, names=RES_COLS)

# Open (or create) the SIFTS DuckDB — SiftsDB() creates the two tables
conn = duckdb.connect(SIFTS_DB)
sdb  = SiftsDB(conn)

# Insert segments
conn.register("tmp_seg", df_seg_load)
conn.execute("INSERT INTO sifts_xref_segment SELECT * FROM tmp_seg")
conn.unregister("tmp_seg")

# Insert residues
conn.register("tmp_res", df_res_load)
conn.execute("INSERT INTO sifts_xref_residue SELECT * FROM tmp_res")
conn.unregister("tmp_res")

print(f"Loaded {len(df_seg_load)} segment(s) and {len(df_res_load)} residue(s) into {SIFTS_DB}")

In [ ]:
# ── Query examples ────────────────────────────────────────────────────────────

# Best segment(s) for 1CBS
df_q_seg = conn.execute("""
    SELECT entry_id, auth_asym_id, accession,
           unp_start, unp_end, pdb_start, pdb_end,
           identity, best_mapping
    FROM   sifts_xref_segment
    WHERE  best_mapping = true
""").df()
print("sifts_xref_segment (best_mapping = True)")
display(df_q_seg)

# First 10 residues of chain A
df_q_res = conn.execute("""
    SELECT pdb_seq_id, unp_seq_id,
           pdb_one_letter_code, unp_one_letter_code,
           type, observed
    FROM   sifts_xref_residue
    WHERE  auth_asym_id = 'A'
    ORDER  BY pdb_seq_id
    LIMIT  10
""").df()
print("\nsifts_xref_residue — first 10 residues of chain A")
display(df_q_res)

conn.close()